# TP3 — Grafos, BFS, DFS e Dijkstra

Notebook resolvido em nível de aluno mediano, com código em inglês e explicações em PT-BR.

## Implementação

As classes abaixo representam grafos não ponderados e ponderados usando lista de adjacência.

In [ ]:

from collections import deque
import heapq


class Graph:
    def __init__(self, directed=False):
        self.directed = directed
        self.adj = {}

    def add_vertex(self, vertex):
        if vertex not in self.adj:
            self.adj[vertex] = []

    def add_edge(self, origin, destination):
        self.add_vertex(origin)
        self.add_vertex(destination)

        if destination not in self.adj[origin]:
            self.adj[origin].append(destination)

        if not self.directed:
            if origin not in self.adj[destination]:
                self.adj[destination].append(origin)

    def print_adj_list(self):
        for vertex in sorted(self.adj):
            print(vertex, ":", sorted(self.adj[vertex]))

    def vertices(self):
        return list(self.adj.keys())

    def dfs(self, start):
        visited = set()
        order = []

        def visit(vertex):
            visited.add(vertex)
            order.append(vertex)

            for neighbor in sorted(self.adj.get(vertex, [])):
                if neighbor not in visited:
                    visit(neighbor)

        visit(start)
        return order

    def bfs(self, start):
        visited = set()
        queue = deque()

        order = []
        visited.add(start)
        queue.append(start)

        while queue:
            vertex = queue.popleft()
            order.append(vertex)

            for neighbor in sorted(self.adj.get(vertex, [])):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        return order

    def has_path(self, start, end):
        if start == end:
            return True

        visited = set()
        queue = deque()

        visited.add(start)
        queue.append(start)

        while queue:
            vertex = queue.popleft()

            for neighbor in self.adj.get(vertex, []):
                if neighbor == end:
                    return True

                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        return False

    def shortest_path_bfs(self, start, end):
        if start == end:
            return [start]

        visited = set()
        queue = deque()

        visited.add(start)
        queue.append((start, [start]))

        while queue:
            vertex, path = queue.popleft()

            for neighbor in sorted(self.adj.get(vertex, [])):
                if neighbor not in visited:
                    new_path = path + [neighbor]

                    if neighbor == end:
                        return new_path

                    visited.add(neighbor)
                    queue.append((neighbor, new_path))

        return []


class WeightedGraph:
    def __init__(self):
        self.adj = {}

    def add_vertex(self, vertex):
        if vertex not in self.adj:
            self.adj[vertex] = []

    def add_edge(self, origin, destination, weight):
        self.add_vertex(origin)
        self.add_vertex(destination)

        self.adj[origin].append((destination, weight))
        self.adj[destination].append((origin, weight))

    def neighbors(self, vertex):
        return self.adj.get(vertex, [])

    def reachable_bfs(self, start):
        visited = set()
        queue = deque()
        order = []

        visited.add(start)
        queue.append(start)

        while queue:
            vertex = queue.popleft()
            order.append(vertex)

            neighbor_names = []
            for neighbor, weight in self.neighbors(vertex):
                neighbor_names.append(neighbor)

            for neighbor in sorted(neighbor_names):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        return order

    def dfs(self, start):
        visited = set()
        order = []

        def visit(vertex):
            visited.add(vertex)
            order.append(vertex)

            neighbor_names = []
            for neighbor, weight in self.neighbors(vertex):
                neighbor_names.append(neighbor)

            for neighbor in sorted(neighbor_names):
                if neighbor not in visited:
                    visit(neighbor)

        visit(start)
        return order

    def shortest_path_bfs(self, start, end):
        visited = set()
        queue = deque()

        visited.add(start)
        queue.append((start, [start]))

        while queue:
            vertex, path = queue.popleft()

            neighbor_names = []
            for neighbor, weight in self.neighbors(vertex):
                neighbor_names.append(neighbor)

            for neighbor in sorted(neighbor_names):
                if neighbor not in visited:
                    new_path = path + [neighbor]

                    if neighbor == end:
                        return new_path

                    visited.add(neighbor)
                    queue.append((neighbor, new_path))

        return []

    def cost_of_path(self, path):
        total = 0

        for i in range(len(path) - 1):
            origin = path[i]
            destination = path[i + 1]
            found = False

            for neighbor, weight in self.neighbors(origin):
                if neighbor == destination:
                    total += weight
                    found = True
                    break

            if not found:
                return None

        return total

    def dijkstra(self, start):
        distances = {}
        previous = {}

        for vertex in self.adj:
            distances[vertex] = float("inf")
            previous[vertex] = None

        distances[start] = 0
        queue = [(0, start)]

        while queue:
            current_distance, vertex = heapq.heappop(queue)

            if current_distance > distances[vertex]:
                continue

            for neighbor, weight in self.neighbors(vertex):
                new_distance = current_distance + weight

                if new_distance < distances[neighbor]:
                    distances[neighbor] = new_distance
                    previous[neighbor] = vertex
                    heapq.heappush(queue, (new_distance, neighbor))

        return distances, previous


def count_teams(number_of_students, friendships):
    graph = Graph()

    for student in range(1, number_of_students + 1):
        graph.add_vertex(student)

    for a, b in friendships:
        graph.add_edge(a, b)

    visited = set()
    groups = 0

    for student in range(1, number_of_students + 1):
        if student not in visited:
            groups += 1
            stack = [student]
            visited.add(student)

            while stack:
                current = stack.pop()

                for neighbor in graph.adj[current]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)

    return groups


def count_valid_tours(number_of_rooms, tunnels, tours):
    graph = Graph()

    for room in range(1, number_of_rooms + 1):
        graph.add_vertex(room)

    for a, b in tunnels:
        graph.add_edge(a, b)

    valid_count = 0

    for tour in tours:
        is_valid = True

        for i in range(len(tour) - 1):
            origin = tour[i]
            destination = tour[i + 1]

            if not graph.has_path(origin, destination):
                is_valid = False
                break

        if is_valid:
            valid_count += 1

    return valid_count


def process_island_operations(number_of_islands, operations):
    graph = Graph()
    answers = []

    for island in range(1, number_of_islands + 1):
        graph.add_vertex(island)

    for operation, a, b in operations:
        if operation == 1:
            graph.add_edge(a, b)
        else:
            if graph.has_path(a, b):
                answers.append(1)
            else:
                answers.append(0)

    return answers


def build_product_graph():
    graph = Graph()
    edges = [
        ("brush", "nail_polish"),
        ("nail_polish", "eye_shadow"),
        ("eye_shadow", "eye_glasses"),
        ("nail_polish", "nails"),
        ("nails", "pins"),
        ("nails", "needles"),
        ("pins", "needles"),
        ("nails", "hammer"),
        ("hammer", "drill"),
        ("hammer", "saw"),
        ("saw", "knife"),
        ("knife", "fork"),
        ("knife", "spoon"),
    ]

    for origin, destination in edges:
        graph.add_edge(origin, destination)

    return graph


def build_social_graph():
    graph = Graph()
    edges = [
        ("Idris", "Kamil"),
        ("Idris", "Talia"),
        ("Kamil", "Lina"),
        ("Lina", "Sasha"),
        ("Sasha", "Marco"),
        ("Marco", "Ken"),
        ("Ken", "Talia"),
    ]

    for origin, destination in edges:
        graph.add_edge(origin, destination)

    return graph


def build_dependency_graph():
    graph = Graph(directed=True)
    edges = [
        ("Inicio", "A"),
        ("Inicio", "B"),
        ("A", "C"),
        ("B", "C"),
        ("C", "D"),
        ("D", "E"),
        ("B", "F"),
        ("F", "E"),
    ]

    for origin, destination in edges:
        graph.add_edge(origin, destination)

    return graph


def build_port_graph():
    graph = WeightedGraph()
    edges = [
        ("Berco_A", "Patio_1", 4),
        ("Berco_A", "Patio_2", 7),
        ("Berco_B", "Patio_2", 3),
        ("Berco_B", "Patio_3", 6),
        ("Patio_1", "Patio_2", 2),
        ("Patio_2", "Patio_3", 2),
        ("Patio_1", "Alfandega", 8),
        ("Patio_2", "Alfandega", 5),
        ("Patio_3", "Centro_Logistico", 4),
        ("Alfandega", "Centro_Logistico", 3),
    ]

    for origin, destination, weight in edges:
        graph.add_edge(origin, destination, weight)

    return graph


def rebuild_path(previous, start, end):
    path = []
    current = end

    while current is not None:
        path.append(current)

        if current == start:
            break

        current = previous[current]

    path.reverse()

    if not path or path[0] != start:
        return []

    return path


def print_distances(distances):
    for vertex in sorted(distances):
        print(vertex, ":", distances[vertex])


def run_basic_tests():
    print("Exercício 1 - Formação de times")
    friendships = [(1, 2), (2, 3), (4, 5)]
    print("Quantidade de grupos:", count_teams(6, friendships), "esperado: 3")

    print("\nExercício 2 - Passeios válidos")
    tunnels = [(1, 2), (2, 3), (4, 5)]
    tours = [
        [1, 3],
        [1, 2, 3],
        [1, 4],
        [4, 5],
    ]
    print("Passeios válidos:", count_valid_tours(5, tunnels, tours), "esperado: 3")

    print("\nExercício 3 - Conectividade dinâmica entre ilhas")
    operations = [
        (0, 1, 2),
        (1, 1, 2),
        (0, 1, 2),
        (1, 2, 3),
        (0, 1, 3),
        (0, 4, 5),
    ]
    print("Respostas:", process_island_operations(5, operations), "esperado: [0, 1, 1, 0]")

    print("\nExercício 4 - DFS em recomendações")
    product_graph = build_product_graph()
    dfs_result = product_graph.dfs("nails")
    print("Ordem DFS:", dfs_result)

    print("\nExercício 5 - BFS em recomendações")
    bfs_result = product_graph.bfs("nails")
    print("Ordem BFS:", bfs_result)

    print("\nExercício 6 - Menor caminho em rede social")
    social_graph = build_social_graph()
    path = social_graph.shortest_path_bfs("Idris", "Lina")
    print("Caminho:", path)
    print("Distância:", len(path) - 1)

    print("\nExercício 7 - Grafo direcionado")
    dependency_graph = build_dependency_graph()
    print("DFS:", dependency_graph.dfs("Inicio"))
    print("BFS:", dependency_graph.bfs("Inicio"))

    print("\nExercício 8 - BFS ignorando pesos na rede logística")
    port_graph = build_port_graph()
    reachable = port_graph.reachable_bfs("Berco_A")
    bfs_path = port_graph.shortest_path_bfs("Berco_A", "Centro_Logistico")
    print("Áreas alcançáveis:", reachable)
    print("Caminho com menor número de etapas:", bfs_path)
    print("Custo desse caminho:", port_graph.cost_of_path(bfs_path))

    print("\nExercício 9 - Dijkstra")
    distances, previous = port_graph.dijkstra("Berco_A")
    print("Menores distâncias a partir de Berco_A:")
    print_distances(distances)

    print("\nExercício 10 - Reconstrução da rota ótima")
    dijkstra_path = rebuild_path(previous, "Berco_A", "Centro_Logistico")
    print("Caminho mínimo:", dijkstra_path)
    print("Custo:", port_graph.cost_of_path(dijkstra_path))

    print("\nExercício 11 - Comparação BFS x Dijkstra")
    print("Caminho BFS:", bfs_path)
    print("Custo BFS:", port_graph.cost_of_path(bfs_path))
    print("Caminho Dijkstra:", dijkstra_path)
    print("Custo Dijkstra:", port_graph.cost_of_path(dijkstra_path))

    print("\nExercício 12 - Análise geral da rede logística")
    print("Ordem DFS:", port_graph.dfs("Berco_A"))
    print("Ordem BFS:", port_graph.reachable_bfs("Berco_A"))
    print("Distâncias Dijkstra:")
    print_distances(distances)


if __name__ == "__main__":
    run_basic_tests()


## Execução dos testes básicos

In [ ]:
run_basic_tests()

# TP3 — Grafos, BFS, DFS e Dijkstra

## Padrão usado na solução

A solução foi feita em um nível de aluno mediano, usando código simples em Python. Os nomes de classes, funções e variáveis foram mantidos em inglês, mas as explicações estão em português.

A base do trabalho é a representação de grafos por lista de adjacência, como nos códigos de referência. Para as partes com peso, foi criada uma classe separada `WeightedGraph`.

---

# Exercício 1 — Formação de times a partir de amizades

## Resolução

O problema foi modelado como um grafo não direcionado. Cada aluno é um vértice e cada amizade é uma aresta. Como amizade é transitiva, cada componente conectada representa um time.

A função implementada foi:

```python
count_teams(number_of_students, friendships)
```

Ela cria o grafo e usa DFS com pilha para contar quantos grupos separados existem.

## Exemplo usado

```python
friendships = [(1, 2), (2, 3), (4, 5)]
count_teams(6, friendships)
```

Resultado:

```text
3 grupos
```

Os grupos são:

```text
{1, 2, 3}, {4, 5}, {6}
```

## Análise e discussão

A DFS visita cada aluno e cada amizade uma vez. Por isso, a complexidade é:

```text
O(N + M)
```

em que `N` é o número de alunos e `M` é o número de amizades.

A escolha por DFS é adequada porque o objetivo é descobrir componentes conectadas, não calcular menor caminho.

---

# Exercício 2 — Validação de passeios em uma rede de túneis

## Resolução

A rede de túneis foi representada como um grafo não direcionado. Para cada sugestão de passeio, o código verifica se cada par consecutivo de salões está conectado por algum caminho.

A função implementada foi:

```python
count_valid_tours(number_of_rooms, tunnels, tours)
```

Para cada par consecutivo, foi usada BFS por meio do método `has_path`.

## Exemplo usado

```python
tunnels = [(1, 2), (2, 3), (4, 5)]
tours = [
    [1, 3],
    [1, 2, 3],
    [1, 4],
    [4, 5],
]
```

Resultado:

```text
3 passeios válidos
```

## Análise e discussão

A BFS verifica se existe caminho entre dois salões. No pior caso, uma busca pode visitar todos os salões e túneis:

```text
O(S + T)
```

Se forem avaliados muitos pares consecutivos dentro das sugestões, o custo pode crescer bastante. Se `L` for o total de pares avaliados em todas as sugestões, o custo fica aproximadamente:

```text
O(L * (S + T))
```

Essa solução é simples e segue diretamente o que foi pedido. Uma versão mais otimizada poderia calcular componentes conectadas uma vez e depois só comparar o componente de cada salão.

---

# Exercício 3 — Conectividade dinâmica entre ilhas

## Resolução

As ilhas foram modeladas como vértices de um grafo não direcionado. As operações são processadas na ordem em que aparecem:

- operação `1 A B`: adiciona uma ponte;
- operação `0 A B`: consulta se existe caminho entre as ilhas.

A função implementada foi:

```python
process_island_operations(number_of_islands, operations)
```

## Exemplo usado

```python
operations = [
    (0, 1, 2),
    (1, 1, 2),
    (0, 1, 2),
    (1, 2, 3),
    (0, 1, 3),
    (0, 4, 5),
]
```

Resultado:

```text
[0, 1, 1, 0]
```

## Análise e discussão

Cada consulta usa BFS para verificar conectividade. No pior caso, uma consulta custa:

```text
O(N + E)
```

Como existem até `M` operações, o pior caso geral pode chegar a:

```text
O(M * (N + M))
```

Essa solução não usa Union-Find porque o enunciado pede busca em grafo com DFS ou BFS. Union-Find seria mais eficiente, mas fugiria um pouco da proposta do exercício.

---

# Exercício 4 — Exploração de recomendações com DFS

## Resolução

O grafo de produtos foi modelado como grafo não direcionado. A DFS começa no vértice `nails`.

A função usada foi:

```python
product_graph.dfs("nails")
```

Os vizinhos são percorridos em ordem lexicográfica para a saída ficar previsível.

## Resultado obtido

```text
['nails', 'hammer', 'drill', 'saw', 'knife', 'fork', 'spoon', 'nail_polish', 'brush', 'eye_shadow', 'eye_glasses', 'needles', 'pins']
```

## Análise e discussão

A DFS aprofunda o caminho antes de voltar para explorar outras alternativas. Por isso, saindo de `nails`, ela segue uma cadeia mais longa antes de retornar para outros vizinhos.

A complexidade é:

```text
O(V + E)
```

porque cada vértice e cada aresta são considerados durante a travessia.

---

# Exercício 5 — Exploração de recomendações com BFS

## Resolução

Foi usado o mesmo grafo de produtos do exercício anterior, mas agora com BFS a partir de `nails`.

A função usada foi:

```python
product_graph.bfs("nails")
```

## Resultado obtido

```text
['nails', 'hammer', 'nail_polish', 'needles', 'pins', 'drill', 'saw', 'brush', 'eye_shadow', 'knife', 'eye_glasses', 'fork', 'spoon']
```

## Análise e discussão

A BFS visita primeiro os vértices mais próximos da origem. Assim, ela pega primeiro os produtos diretamente ligados a `nails`, depois os produtos a distância 2, e assim por diante.

A diferença principal é:

- DFS aprofunda caminhos;
- BFS percorre por camadas.

A complexidade também é:

```text
O(V + E)
```

---

# Exercício 6 — Menor caminho em rede social

## Resolução

A rede social foi representada como grafo não direcionado. Como o grafo não tem peso, a BFS garante o menor caminho em número de arestas.

A função usada foi:

```python
social_graph.shortest_path_bfs("Idris", "Lina")
```

## Resultado obtido

```text
['Idris', 'Kamil', 'Lina']
```

Distância:

```text
2 conexões
```

## Análise e discussão

A BFS encontra o menor caminho em grafos não ponderados porque visita primeiro todos os vértices a distância 1, depois todos a distância 2, e assim por diante.

Quando `Lina` é encontrada, o caminho achado já é mínimo em número de conexões.

A complexidade é:

```text
O(V + E)
```

---

# Exercício 7 — Travessia em grafo direcionado

## Resolução

O grafo de dependências foi modelado como grafo direcionado. Isso significa que uma aresta de `A` para `C` não permite automaticamente voltar de `C` para `A`.

Foram executadas DFS e BFS a partir de `Inicio`.

## Resultado obtido

DFS:

```text
['Inicio', 'A', 'C', 'D', 'E', 'B', 'F']
```

BFS:

```text
['Inicio', 'A', 'B', 'C', 'F', 'D', 'E']
```

## Análise e discussão

A diferença ocorre porque a DFS aprofunda primeiro por `A`, depois segue para `C`, `D` e `E`. Já a BFS visita primeiro os vizinhos diretos de `Inicio`, que são `A` e `B`.

A direção das arestas limita os caminhos possíveis. Por exemplo, se existe `Inicio -> A`, isso não significa que existe caminho de `A` para `Inicio`.

A complexidade dos dois algoritmos é:

```text
O(V + E)
```

---

# Exercício 8 — Viabilidade operacional na movimentação de containers

## Resolução

A rede logística foi modelada como grafo ponderado, usando a classe `WeightedGraph`. Cada aresta possui um custo operacional.

Primeiro foi feita uma BFS ignorando os pesos, para descobrir áreas alcançáveis e o caminho com menor número de etapas entre `Berco_A` e `Centro_Logistico`.

## Resultado obtido

Áreas alcançáveis:

```text
['Berco_A', 'Patio_1', 'Patio_2', 'Alfandega', 'Berco_B', 'Patio_3', 'Centro_Logistico']
```

Caminho com menor número de etapas:

```text
['Berco_A', 'Patio_1', 'Alfandega', 'Centro_Logistico']
```

Custo desse caminho:

```text
15
```

## Análise e discussão

A BFS ignora os pesos. Ela considera apenas quantidade de arestas. Por isso, o caminho encontrado é bom em número de etapas, mas não necessariamente tem menor custo operacional.

A complexidade da BFS é:

```text
O(V + E)
```

---

# Exercício 9 — Menor custo com Dijkstra

## Resolução

Foi aplicado o algoritmo de Dijkstra na rede logística, usando `Berco_A` como origem.

A função usada foi:

```python
port_graph.dijkstra("Berco_A")
```

Ela retorna dois dicionários:

- `distances`: menor custo até cada vértice;
- `previous`: predecessor usado para reconstruir o caminho.

## Resultado obtido

```text
Alfandega : 11
Berco_A : 0
Berco_B : 9
Centro_Logistico : 12
Patio_1 : 4
Patio_2 : 6
Patio_3 : 8
```

## Análise e discussão

Dijkstra considera os pesos das arestas. Por isso, ele é adequado para encontrar menor custo em grafos com pesos não negativos.

Como foi usada fila de prioridade com `heapq`, a complexidade fica aproximadamente:

```text
O((V + E) log V)
```

---

# Exercício 10 — Reconstrução de rota ótima

## Resolução

A reconstrução da rota usa o dicionário `previous`, gerado pelo Dijkstra. A função começa no destino e volta pelos predecessores até chegar na origem.

Função usada:

```python
rebuild_path(previous, "Berco_A", "Centro_Logistico")
```

## Resultado obtido

```text
['Berco_A', 'Patio_1', 'Patio_2', 'Patio_3', 'Centro_Logistico']
```

Custo:

```text
12
```

## Análise e discussão

A navegação é feita ao contrário:

```text
Centro_Logistico <- Patio_3 <- Patio_2 <- Patio_1 <- Berco_A
```

Depois a lista é invertida para mostrar o caminho da origem até o destino.

A complexidade da reconstrução depende do tamanho do caminho:

```text
O(k)
```

em que `k` é a quantidade de vértices na rota.

---

# Exercício 11 — Impacto dos pesos na escolha de rotas

## Resolução

Foram comparados dois caminhos entre `Berco_A` e `Centro_Logistico`:

1. caminho encontrado por BFS ignorando pesos;
2. caminho encontrado por Dijkstra considerando pesos.

## Resultado obtido

BFS:

```text
['Berco_A', 'Patio_1', 'Alfandega', 'Centro_Logistico']
Custo: 15
```

Dijkstra:

```text
['Berco_A', 'Patio_1', 'Patio_2', 'Patio_3', 'Centro_Logistico']
Custo: 12
```

## Análise e discussão

O caminho da BFS tem menos etapas, mas maior custo. O Dijkstra escolhe uma rota com mais arestas, porém menor custo total.

Isso mostra que BFS não serve para otimização de custo quando as arestas possuem pesos diferentes. Nesses casos, o algoritmo correto é Dijkstra, desde que os pesos não sejam negativos.

---

# Exercício 12 — Análise de rotas na rede logística

## Resolução

Foram aplicados DFS, BFS e Dijkstra a partir de `Berco_A`.

Resultados:

DFS:

```text
['Berco_A', 'Patio_1', 'Alfandega', 'Centro_Logistico', 'Patio_3', 'Berco_B', 'Patio_2']
```

BFS:

```text
['Berco_A', 'Patio_1', 'Patio_2', 'Alfandega', 'Berco_B', 'Patio_3', 'Centro_Logistico']
```

Dijkstra:

```text
Alfandega : 11
Berco_A : 0
Berco_B : 9
Centro_Logistico : 12
Patio_1 : 4
Patio_2 : 6
Patio_3 : 8
```

## Análise e discussão

A DFS aprofunda um caminho antes de voltar. Ela é útil para explorar estrutura, verificar conectividade e percorrer componentes.

A BFS percorre em camadas. Ela é boa para encontrar menor caminho em quantidade de arestas, quando o grafo não tem pesos.

O Dijkstra considera os custos. Ele é melhor quando o objetivo é menor custo total, como no exemplo da rede logística.

Resumo:

| Algoritmo | Melhor uso | Limitação |
|---|---|---|
| DFS | exploração e conectividade | não garante menor caminho |
| BFS | menor caminho em grafo não ponderado | ignora custos diferentes |
| Dijkstra | menor custo em grafo ponderado | exige pesos não negativos |

---

# Conclusão

O TP3 mostra que diferentes problemas em grafos pedem algoritmos diferentes. DFS e BFS resolvem bem conectividade e travessias. BFS também encontra menor caminho quando todas as arestas têm o mesmo peso. Já Dijkstra deve ser usado quando existe custo diferente em cada conexão.
